## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

#### ARCHITECTURE 3: Modern Local Transformer (RoPE + GQA + Sliding Window + RMSNorm + SwiGLU + KV Cache)


In [ ]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# Phase-1 files
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 2048

# Architecture 3: Modern local causal Transformer
N_EMBD = 600
N_LAYER = 8
N_HEAD = 10
N_KV_HEAD = 5          # 2 query heads share each K/V head
FFN_DIM = 1600         # SwiGLU hidden size
LOCAL_WINDOW = 512
ATTN_CHUNK_SIZE = 256  # keeps local-attention training memory practical
DROPOUT = 0.1

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 500
TEMPERATURE = 0.4
TOP_P = 0.95

# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1_a3_modern_transformer"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "architecture-3-final")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert N_EMBD % N_HEAD == 0
assert N_HEAD % N_KV_HEAD == 0

print("Configuration loaded.")
print(f"Head dimension: {N_EMBD // N_HEAD}")
print(f"Query heads: {N_HEAD}, KV heads: {N_KV_HEAD}")


## 4. Load Tokenizer

In [ ]:
from transformers import PreTrainedTokenizerFast
import os

DATA_DIR = "/kaggle/input/datasets/srivenkatm/phase1-simple-data/phase1_simple_data"

TOKENIZER_PATH = os.path.join(DATA_DIR, "tokenizer.json")
TRAIN_BIN = os.path.join(DATA_DIR, "train_tokens.bin")
VAL_BIN = os.path.join(DATA_DIR, "val_tokens.bin")

print("Tokenizer exists:", os.path.exists(TOKENIZER_PATH))
print("Train tokens exist:", os.path.exists(TRAIN_BIN))
print("Val tokens exist:", os.path.exists(VAL_BIN))

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=TOKENIZER_PATH,
    pad_token="<pad>",
    unk_token="<unk>",
    bos_token="<s>",
    eos_token="</s>"
)

print("Vocabulary size:", len(tokenizer))

## 5. Download Training Data From Hugging Face

In [ ]:

# ------------------------------------------------------------
# The tokenizer notebook has ALREADY prepared exactly:
#   200M training token IDs
#   5M validation token IDs
#
# Therefore we do not download and tokenize the raw training JSON again.
# We directly reuse the saved .bin files.
# ------------------------------------------------------------

TRAIN_TOKEN_FILE = os.path.join(DATA_DIR, "train_tokens.bin")
VAL_TOKEN_FILE = os.path.join(DATA_DIR, "val_tokens.bin")

print("Training token file:")
print(TRAIN_TOKEN_FILE)

print("Validation token file:")
print(VAL_TOKEN_FILE)


## 6. Create Train and Validation Tokens Set

In [ ]:

import numpy as np

# ------------------------------------------------------------
# Memory-map the token files.
#
# np.memmap is useful here because 200M token IDs are large.
# It reads small pieces from disk only when they are needed,
# instead of loading the whole file into RAM.
#
# uint16 is enough because our vocabulary is about 50k,
# and 50,000 < 65,536.
# ------------------------------------------------------------

train_tokens = np.memmap(
    TRAIN_TOKEN_FILE,
    dtype=np.uint16,
    mode="r",
    shape=(TRAIN_TOKENS,)
)

val_tokens = np.memmap(
    VAL_TOKEN_FILE,
    dtype=np.uint16,
    mode="r",
    shape=(VAL_TOKENS,)
)

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

# ------------------------------------------------------------
# Safety check 1: uint16 can store token IDs only from 0 to 65,535.
# Therefore the tokenizer vocabulary must fit inside uint16.
# ------------------------------------------------------------
UINT16_MAX_ID = np.iinfo(np.uint16).max
assert len(tokenizer) <= UINT16_MAX_ID + 1, (
    f"Tokenizer vocabulary has {len(tokenizer):,} entries, but uint16 can "
    f"represent at most {UINT16_MAX_ID + 1:,} distinct token IDs."
)
print(f"uint16 vocabulary check passed: vocab_size={len(tokenizer):,}")

# ------------------------------------------------------------
# Safety check 2: scan ALL saved token IDs and verify that every ID
# is valid for the tokenizer currently loaded above.
#
# We scan in chunks so the complete 200M-token file does not need
# to be copied into RAM at once.
# ------------------------------------------------------------
def scan_token_ids(token_array, name, vocab_size, chunk_size=5_000_000):
    min_id = UINT16_MAX_ID
    max_id = 0

    for start in range(0, len(token_array), chunk_size):
        chunk = token_array[start:start + chunk_size]
        if len(chunk) == 0:
            continue

        chunk_min = int(chunk.min())
        chunk_max = int(chunk.max())

        min_id = min(min_id, chunk_min)
        max_id = max(max_id, chunk_max)

    assert min_id >= 0
    assert max_id < vocab_size, (
        f"{name} contains token ID {max_id:,}, but the loaded tokenizer "
        f"has vocabulary size {vocab_size:,}. The .bin files and tokenizer "
        f"do not match."
    )

    print(
        f"{name} full scan passed: "
        f"min_id={min_id:,}, max_id={max_id:,}, vocab_size={vocab_size:,}"
    )


scan_token_ids(train_tokens, "Training tokens", len(tokenizer))
scan_token_ids(val_tokens, "Validation tokens", len(tokenizer))


## 7. Create Token Blocks

In [ ]:

from torch.utils.data import Dataset

class TokenBlockDataset(Dataset):
    """
    Read one fixed-size block directly from the long token file.

    Example with block_size = 4:
        token file: [10, 20, 30, 40, 50, 60, 70, 80]

        block 0 -> [10, 20, 30, 40]
        block 1 -> [50, 60, 70, 80]

    We do not create a huge Python list of all blocks.
    """

    def __init__(self, tokens, block_size):
        self.tokens = tokens
        self.block_size = block_size
        self.num_blocks = len(tokens) // block_size

    def __len__(self):
        return self.num_blocks

    def __getitem__(self, index):
        start = index * self.block_size
        end = start + self.block_size

        # Convert only this block to a normal integer tensor.
        block = np.array(
            self.tokens[start:end],
            dtype=np.int64
        )

        return {
            "input_ids": torch.tensor(
                block,
                dtype=torch.long
            )
        }

print("Token block dataset class created.")


## 8. Create Datasets

In [ ]:

train_dataset = TokenBlockDataset(
    train_tokens,
    TRAIN_BLOCK_SIZE
)

val_dataset = TokenBlockDataset(
    val_tokens,
    TRAIN_BLOCK_SIZE
)

print("Training sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))


## 9. Define The Model

#### ARCHITECTURE 3: Modern Local Transformer (RoPE + GQA + Sliding Window + RMSNorm + SwiGLU + KV Cache)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.modeling_outputs import CausalLMOutput

# ------------------------------------------------------------
# ARCHITECTURE 3: MODERN LOCAL CAUSAL TRANSFORMER
#
# Main changes from a vanilla GPT-style decoder:
#   1. RoPE instead of learned absolute positional embeddings
#   2. Grouped-Query Attention (10 query heads, 5 K/V heads)
#   3. Sliding-window causal self-attention
#   4. RMSNorm instead of LayerNorm
#   5. SwiGLU instead of GELU FFN
#   6. KV cache during autoregressive generation
#
# This is implemented from scratch with PyTorch modules.
# No GPT-2 model/config/weights are used.
# ------------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Compute the RMS in fp32 for numerical stability, then cast back.
        rms = x.float().pow(2).mean(dim=-1, keepdim=True)
        x_norm = x * torch.rsqrt(rms + self.eps).to(x.dtype)
        return x_norm * self.weight


def rotate_half(x):
    # RoPE rotates each consecutive pair: (x0, x1) -> (-x1, x0).
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE needs an even head dimension."
        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)
        )
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def cos_sin(self, positions, dtype):
        # positions: [seq]
        freqs = torch.outer(positions.float(), self.inv_freq)
        cos = freqs.cos().repeat_interleave(2, dim=-1).to(dtype)
        sin = freqs.sin().repeat_interleave(2, dim=-1).to(dtype)
        return cos[None, None, :, :], sin[None, None, :, :]

    def apply_rotary(self, q, k, positions):
        cos, sin = self.cos_sin(positions, q.dtype)
        q = (q * cos) + (rotate_half(q) * sin)
        k = (k * cos) + (rotate_half(k) * sin)
        return q, k


class GroupedQueryLocalAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        window_size,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        assert dim % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.groups = num_heads // num_kv_heads
        self.window_size = window_size
        self.chunk_size = chunk_size
        self.dropout = dropout

        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)

        self.rope = RotaryEmbedding(self.head_dim)

    def _repeat_kv(self, x):
        # [B, kv_heads, T, D] -> [B, query_heads, T, D]
        return x.repeat_interleave(self.groups, dim=1)

    def _local_attention(self, q, k, v):
        """
        Exact causal sliding-window attention.

        We process a few query positions at a time.  Each query chunk only
        materializes the K/V region that can actually be attended to, instead
        of creating a full 2048 x 2048 attention matrix.
        """
        B, H, T, D = q.shape
        outputs = []

        k_rep = self._repeat_kv(k)
        v_rep = self._repeat_kv(v)

        for q_start in range(0, T, self.chunk_size):
            q_end = min(q_start + self.chunk_size, T)

            # Earliest key any query in this chunk could legally use.
            k_start = max(0, q_start - self.window_size + 1)
            k_end = q_end

            q_chunk = q[:, :, q_start:q_end, :]
            k_chunk = k_rep[:, :, k_start:k_end, :]
            v_chunk = v_rep[:, :, k_start:k_end, :]

            q_pos = torch.arange(q_start, q_end, device=q.device)
            k_pos = torch.arange(k_start, k_end, device=q.device)

            # True = allowed attention position.
            causal = k_pos[None, :] <= q_pos[:, None]
            within_window = k_pos[None, :] >= (
                q_pos[:, None] - self.window_size + 1
            )
            attn_mask = (causal & within_window)[None, None, :, :]

            out = F.scaled_dot_product_attention(
                q_chunk,
                k_chunk,
                v_chunk,
                attn_mask=attn_mask,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=False,
            )
            outputs.append(out)

        return torch.cat(outputs, dim=2)

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T, _ = x.shape

        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=x.device,
        )
        q, k = self.rope.apply_rotary(q, k, positions)

        if past_kv is None:
            # Training/prefill path: causal local attention over this sequence.
            attn_out = self._local_attention(q, k, v)
            k_all, v_all = k, v
        else:
            # Generation path.  We decode one token at a time after prefill.
            assert T == 1, "Cached decoding expects one new token at a time."
            past_k, past_v = past_kv
            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)

            # Keep only the most recent local window in the KV cache.
            k_all = k_all[:, :, -self.window_size:, :]
            v_all = v_all[:, :, -self.window_size:, :]

            k_rep = self._repeat_kv(k_all)
            v_rep = self._repeat_kv(v_all)

            # Every cached token is in the past (or the current token), so no
            # extra causal mask is necessary for this one-token decode step.
            attn_out = F.scaled_dot_product_attention(
                q,
                k_rep,
                v_rep,
                dropout_p=0.0,
                is_causal=False,
            )

        new_kv = None
        if use_cache:
            new_kv = (
                k_all[:, :, -self.window_size:, :],
                v_all[:, :, -self.window_size:, :],
            )

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.dim)
        return self.o_proj(attn_out), new_kv


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class ModernTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        ffn_dim,
        window_size,
        chunk_size,
        dropout,
    ):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = GroupedQueryLocalAttention(
            dim=dim,
            num_heads=num_heads,
            num_kv_heads=num_kv_heads,
            window_size=window_size,
            chunk_size=chunk_size,
            dropout=dropout,
        )
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, ffn_dim, dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, position_offset=0):
        attn_out, new_kv = self.attn(
            self.attn_norm(x),
            past_kv=past_kv,
            use_cache=use_cache,
            position_offset=position_offset,
        )
        x = x + self.resid_dropout(attn_out)
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv


class ModernLocalTransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim=600,
        num_layers=8,
        num_heads=10,
        num_kv_heads=5,
        ffn_dim=1600,
        window_size=512,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim
        self.num_layers = num_layers

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.embed_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            ModernTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                num_kv_heads=num_kv_heads,
                ffn_dim=ffn_dim,
                window_size=window_size,
                chunk_size=chunk_size,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Weight tying saves parameters and is standard for causal LMs.
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_offset=0,
    ):
        x = self.embed_dropout(self.token_embedding(input_ids))
        new_caches = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, new_kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                position_offset=position_offset,
            )
            if use_cache:
                new_caches.append(new_kv)

        x = self.final_norm(x)
        return x, new_caches

    def forward(self, input_ids, labels=None, attention_mask=None, **kwargs):
        x, _ = self._run_hidden(input_ids)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            # Position t predicts token t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(loss=loss, logits=logits)

    @torch.inference_mode()
    def prefill(self, input_ids):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=None,
            use_cache=True,
            position_offset=0,
        )
        return self.lm_head(x), caches

    @torch.inference_mode()
    def decode_step(self, input_ids, past_key_values, position_offset):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            position_offset=position_offset,
        )
        return self.lm_head(x[:, -1, :]), caches


model = ModernLocalTransformerLM(
    vocab_size=len(tokenizer),
    dim=N_EMBD,
    num_layers=N_LAYER,
    num_heads=N_HEAD,
    num_kv_heads=N_KV_HEAD,
    ffn_dim=FFN_DIM,
    window_size=LOCAL_WINDOW,
    chunk_size=ATTN_CHUNK_SIZE,
    dropout=DROPOUT,
)

model_size = sum(p.numel() for p in model.parameters())
print(f"Model size: {model_size / 1e6:.2f}M parameters")
assert 50_000_000 <= model_size <= 300_000_000


In [ ]:

def data_collator(features):
    """
    Stack equal-length token blocks into one batch.

    For causal language modelling:
        input_ids = [t0, t1, t2, ...]
        labels    = [t0, t1, t2, ...]

    The model internally shifts them by one position when computing
    next-token prediction loss.
    """
    input_ids = torch.stack([
        item["input_ids"]
        for item in features
    ])

    return {
        "input_ids": input_ids,
        "labels": input_ids.clone()
    }


## 10. Training Arguments

In [ ]:

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=EPOCHS,

    # Smaller physical batch is safer for free Kaggle GPUs.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    # 8 small batches are accumulated before one optimizer update.
    gradient_accumulation_steps=8,

    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=500,
    lr_scheduler_type="cosine",

    # Same mixed-precision idea as the starter notebook.
    fp16=torch.cuda.is_available(),

    eval_strategy="steps",
    eval_steps=1000,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,

    # Required for this custom model because token embedding and LM head
    # share the same weights; safetensors rejects that shared storage.
    save_safetensors=False,
    seed=SEED,
    report_to="none",
    logging_dir=os.path.join(OUTPUT_DIR, "logs")
)


## 11. Training

In [ ]:

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

start_time = time.time()

trainer.train()

training_time = (time.time() - start_time)

tokens_per_epoch = (len(train_dataset) * TRAIN_BLOCK_SIZE)
total_tokens = (tokens_per_epoch * EPOCHS)
tokens_per_second = (total_tokens / training_time)

print(f"Training time: {training_time / 3600:.2f} hours")
print(f"Training speed: {tokens_per_second:,.0f} tokens/sec")


## 12. Perplexity

In [ ]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {perplexity:.4f}")

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps = []
train_loss = []
eval_steps = []
eval_loss = []

for log in logs:
    if ("loss" in log and "step" in log):
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if ("eval_loss" in log and "step" in log):
        eval_steps.append(log["step"])
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Training Loss")
plt.plot(eval_steps, eval_loss, label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Phase 1 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:")
print(LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
# Save the custom model weights, tokenizer, and architecture config.
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(MODEL_OUTPUT_DIR, "model_state.pt")
)

architecture_config = {
    "architecture": "ModernLocalTransformerLM",
    "vocab_size": len(tokenizer),
    "dim": N_EMBD,
    "num_layers": N_LAYER,
    "num_heads": N_HEAD,
    "num_kv_heads": N_KV_HEAD,
    "ffn_dim": FFN_DIM,
    "window_size": LOCAL_WINDOW,
    "chunk_size": ATTN_CHUNK_SIZE,
    "dropout": DROPOUT,
}
with open(os.path.join(MODEL_OUTPUT_DIR, "architecture_config.json"), "w") as f:
    json.dump(architecture_config, f, indent=2)

tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("Final model, config, and tokenizer saved to:")
print(MODEL_OUTPUT_DIR)


## 15. Push Model to Hugging Face Hub

In [ ]:
# model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
from transformers import PreTrainedTokenizerFast
import torch

generation_tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_OUTPUT_DIR)

with open(os.path.join(MODEL_OUTPUT_DIR, "architecture_config.json"), "r") as f:
    saved_config = json.load(f)

generation_model = ModernLocalTransformerLM(
    vocab_size=len(generation_tokenizer),
    dim=saved_config["dim"],
    num_layers=saved_config["num_layers"],
    num_heads=saved_config["num_heads"],
    num_kv_heads=saved_config["num_kv_heads"],
    ffn_dim=saved_config["ffn_dim"],
    window_size=saved_config["window_size"],
    chunk_size=saved_config["chunk_size"],
    dropout=saved_config["dropout"],
)

state_dict = torch.load(
    os.path.join(MODEL_OUTPUT_DIR, "model_state.pt"),
    map_location="cpu",
    weights_only=True,
)
generation_model.load_state_dict(state_dict)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()

print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")


## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
def sample_top_p(logits, temperature=0.4, top_p=0.95):
    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)

    sorted_probs, sorted_ids = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)

    remove = cumulative > top_p
    remove[:, 1:] = remove[:, :-1].clone()
    remove[:, 0] = False

    sorted_probs = sorted_probs.masked_fill(remove, 0.0)
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

    sampled_index = torch.multinomial(sorted_probs, 1)
    return sorted_ids.gather(1, sampled_index)


def generate_batch(prompts, model, tokenizer, device, target_words=128):
    """
    Autoregressive generation with a real KV cache.

    A3 keeps only the most recent LOCAL_WINDOW K/V states in every layer,
    matching the sliding-window attention used during training.
    """
    predictions = []

    for prompt in prompts:
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)

        # Reserve room for generation.  RoPE itself can extrapolate, but keeping
        # the same 2048-token working budget makes the comparison with A1/A2 fair.
        max_prompt_tokens = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
        prompt_ids = prompt_ids[-max_prompt_tokens:]

        x = torch.tensor([prompt_ids], dtype=torch.long, device=device)
        generated_ids = []

        with torch.inference_mode():
            # Prefill the prompt once and cache K/V for all 8 layers.
            prompt_logits, past_key_values = model.prefill(x)

            # The final prompt position predicts the first continuation token.
            logits = prompt_logits[:, -1, :]

            for step in range(MAX_GENERATION_TOKENS):
                next_token = sample_top_p(
                    logits,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                )
                generated_ids.append(int(next_token.item()))

                generated_text = tokenizer.decode(
                    generated_ids,
                    skip_special_tokens=True,
                )
                if len(generated_text.split()) >= target_words:
                    break

                # The token just sampled occupies this absolute RoPE position.
                position = len(prompt_ids) + step
                logits, past_key_values = model.decode_step(
                    next_token,
                    past_key_values=past_key_values,
                    position_offset=position,
                )

        words = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ).split()

        if len(words) < target_words:
            raise RuntimeError(
                f"Generation produced only {len(words)} words before the "
                f"{MAX_GENERATION_TOKENS}-token limit. Increase MAX_GENERATION_TOKENS."
            )

        predictions.append(" ".join(words[:target_words]))

    return predictions


## 20. Generate Predictions for the Test Set

In [ ]:
predictions = {}
GENERATION_BATCH_SIZE = 32

items = list(test_data.items())
total_examples = len(items)
print(f"Total test examples: {total_examples}")

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_ids = [str(example_id) for example_id, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]

    batch_predictions = generate_batch(
        prompts=batch_prompts,
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=128
    )

    for example_id, prediction in zip(batch_ids, batch_predictions):
        predictions[example_id] = {"text": prediction}

    completed = start + len(batch)
    print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

## 21. Save prediction.json

In [ ]:
# ------------------------------------------------------------
# Validate the complete Phase-1 submission BEFORE writing prediction.json.
# Required by the capstone:
#   - every official test ID must be present
#   - no unknown/extra IDs
#   - every text must be non-empty
#   - every continuation must contain exactly 128 whitespace-separated words
# ------------------------------------------------------------
official_ids = {str(example_id) for example_id in test_data.keys()}
predicted_ids = set(predictions.keys())

missing_ids = official_ids - predicted_ids
extra_ids = predicted_ids - official_ids

assert not missing_ids, f"Missing official test IDs: {sorted(missing_ids)[:10]}"
assert not extra_ids, f"Unknown/extra prediction IDs: {sorted(extra_ids)[:10]}"

for example_id in official_ids:
    record = predictions[example_id]

    assert isinstance(record, dict), (
        f"Prediction for ID {example_id} must be an object/dict."
    )
    assert "text" in record, f"Prediction for ID {example_id} has no 'text' field."

    text = record["text"]
    assert isinstance(text, str) and text.strip(), (
        f"Prediction text for ID {example_id} is empty."
    )

    word_count = len(text.split())
    assert word_count == TARGET_GENERATION_WORDS, (
        f"ID {example_id} has {word_count} whitespace-separated words; "
        f"expected exactly {TARGET_GENERATION_WORDS}."
    )

print(
    f"Submission validation passed: {len(predictions)} IDs, "
    f"all continuations have exactly {TARGET_GENERATION_WORDS} words."
)

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)